# Лабораторная работа 9

# Создание текстового файла большого объема с помощью генераторной функции

Немчинович Виктория Дмитриевна, 30.05.2025

### Постановка задачи

Напишите генераторную функцию для создания текстового файла c именем `file_name` размера `file_size` в мегабайтах, строки которого будут состоять из случайного количества слов из диапазона `number_words` случайной длины `number_symbols` и составленных из случайных символов заданного типа `type_symbols`. Слова в строке разделены одним пробелом.

### Функция randint из модуля random

Для выполнения заданий лабораторной работы будем использовать функцию генерации целых чисел `randint` из модуля `random` для создания:
- случайного количества слов в строке файла по заданному диапазону
`number_words`,
- случайной длины слова в строке файл по заданному диапазону
`number_symbols`,
- случайного символа в слове из коллекции `type_symbols`.

In [49]:
from random import randint

In [50]:
?randint

Signature: randint(a, b)
Docstring:
Return random integer in range [a, b], including both end points.
        
File:      c:\programdata\anaconda3\envs\widgets_env\lib\random.py
Type:      method

Приведем пример создания символа прописной латинской буквы по его коду,
сгенерированному случайно:

In [51]:
latin_code = (ord('a'), ord('z')) # соответствует type_symbols = 'latin'
f'Код символа: {(c:=randint(*latin_code))}, символ: {chr(c)}'

'Код символа: 106, символ: j'

### Функция process_time из модуля time

Для выполнения заданий лабораторной работы будем использовать функцию
`process_time` из модуля `time`, которая возвращает значение времени,
затраченное процесором на выполнение кода

In [52]:
from time import process_time

In [53]:
?process_time

Docstring:
process_time() -> float

Process time for profiling: sum of the kernel and user-space CPU time.
Type:      builtin_function_or_method

Пример использования функции `process_time`:

In [54]:
start = process_time()
[x**2 for x in range(1_000_000)]
end = process_time()
print(f'{end - start} секунд')

0.40625 секунд


## Задание 9.1. Запись данных в текстовый файл

Откроем файл с именем `test.txt` в текстовом режиме для записи

In [55]:
file_name = 'test.txt'
f = open(file_name, 'w')

Создадим список, состоящий из некоторых строковых объектов, каждый из которых
заканчивается символом новой строки `\n`

In [56]:
lines_list = [f'{x}\n' for x in 'test']
lines_list

['t\n', 'e\n', 's\n', 't\n']

Запишем созданные строки в файл с помощью метода `writelines` файлового
объекта `f`

In [57]:
f.writelines(lines_list)

Вызовем метод `close` файлового объекта `f` для прекращения его связи с
внешним файлом, освобождения системных ресурсов и сброса буферизованного
вывода на диск, если он находится в памяти.

In [58]:
f.close()

Прочитаем записанные данные. Для этого откроем файл в режиме для чтения `'r'`

In [59]:
with open(file_name,'r') as f:
    for line in f:
        print(line, end='')

t
e
s
t


В случае записи в файл больших объемов данных целесообразнее использовать не
списки для хранения строк, а генераторные объекты

In [60]:
lines_gen_expr = (f'{x}\n' for x in range(10**6))

def lines_gen_fun(number_lines=10**6):
    yield from (f'{x}\n' for x in range(number_lines))
    
with open(file_name,'w') as f:
    f.writelines(lines_gen_expr)
    f.writelines(lines_gen_fun())

**В дальнейшем будем создавать данные для записи в файл с помощью
генераторной функции.**

Добавим в функцию `lines_gen_fun` подсчет длины записанной информации в мегабайтах. Полагаем, что длина одного символа равна одному байту

In [61]:
def lines_gen_fun(number_lines = 10**6):
    file_size = 0
    for x in range(number_lines):
        line = f'{x}\n'
        yield line
        file_size += len(line)
    print(f'{file_size/1024**2} Mb')

with open(file_name,'w') as f:
    f.writelines(lines_gen_fun())

6.569757461547852 Mb


**Комментарий к коду.** Допущение, что размер символа в строке равен одному
байту, не всегда верно. В частности, для кириллических символов это не так. Более
правильным решением для подсчета длины записанной информации в байтах будет
использование метода `encode` для записываемых строк. В этом случае в коде
выше вместо `len(line)` стоит вызывать `len(line.encode('utf8'))`.

In [62]:
len('f'), len('f'.encode('utf8')), len('ц'), len('ц'.encode('utf8'))

(1, 1, 1, 2)

Измеряем время, необходимое для создания файла с помощью функции
`lines_gen_fun`

In [63]:
with open(file_name,'w') as f:
    start = process_time()
    f.writelines(lines_gen_fun())
    end = process_time()
    print(end - start, 'секунд')

6.569757461547852 Mb
1.6875 секунд


Внесем изменения в генераторную функцию `lines_gen_fun`, чтобы она
отображала процент записанных данных в файл, если аргумент `status=True`. Это
существенно увеличит время выполнения функции!

In [64]:
def lines_gen_fun(number_lines=10**6, status=False):
    file_size = 0  # переменная для подсчёта общего размера данных
    for x in range(number_lines):  # цикл генерации строк
        line = f'{x}\n'  # строка данных с номером и переводом строки
        yield line  # возврат сгенерированной строки
        file_size += len(line)  # подсчёт общего размера записанных данных в байтах
        # отображение статуса записи в файл в процентах
        if status:
            status_number = x/number_lines*100  # вычисление процента выполнения
            # \r возврат курсора в начало строки для перезаписи
            print(f'\r{int(status_number)}%', end='', flush=True)  # вывод прогресса(flush=True--немедленный вывод)
    print(f'\n {file_size/1024**2} Mb')  # вывод размера данных в Mb после завершения

Запишем данные в файл:

In [78]:
with open(file_name,'w') as f:
    start = process_time()
    f.writelines(lines_gen_fun(3*10**3, status=True))
    end = process_time()
    print(f'{end - start} секунд')

99%
 0.013246536254882812 Mb
3.40625 секунд


## Задание 9.2. Генерация строк

Определим значения переменных, которые задают параметры создания строки для
записи в файл

In [66]:
latin_code = (ord('a'), ord('z')) # соответствует type_symbols = 'latin'
number_words = (10, 20)
number_symbols = (5, 10)

Создадим одно слово случайной длины из интервала `[5,10]`, состоящее из
случайно выбранных латинских символов

In [67]:
len_word = randint(*number_symbols)
word = ''.join([chr(randint(*latin_code)) for i in range(len_word)])
word

'fzxlmje'

Напишем и протестируем код для генерации строки со случайным количеством слов,
случайной длиной слова и случайным набором символов в слове для трех
множеств смволов: прописные латинские символы, цифровые символы,
прописные кириллические символы.

In [68]:
import random
from random import randint

def generate_random_word(length, char_set):
    chars = {
        'latin': 'abcdefghijklmnopqrstuvwxyz',
        'digits': '0123456789',
        'cyrillic': 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя'
    }
    return ''.join(random.choice(chars[char_set]) for _ in range(length))

def generate_random_sentence(num_words, len_range, char_set):
    return ' '.join(
        generate_random_word(randint(*len_range), char_set)
        for _ in range(num_words)
    )

print("Латинские символы:", generate_random_sentence(5, (3, 10), 'latin'))
print("Цифры:", generate_random_sentence(3, (2, 5), 'digits'))
print("Кириллица:", generate_random_sentence(4, (4, 8), 'cyrillic'))

Латинские символы: febkn ovyoxsi ujyz sdqazjibot zpxmyvm
Цифры: 692 28 44441
Кириллица: ячьяъбёя чфёьт чижоцэ фббъкыи


Напишем генераторную функцию `lines_gen_fun_v2(file_size,
type_symbols, number_words, number_symbols)` для генерации строк,
суммарный размер которых в мегабайтах равен `file_size`, а также строки документации для неёё

In [69]:
def lines_gen_fun_v2(file_size, type_symbols, number_words, number_symbols):
    """
    Генераторная функция для создания строк заданного размера
    
    Параметры:
        file_size (float): целевой размер файла в мегабайтах
        type_symbols (str): тип символов ('latin', 'digits' или 'cyrillic')
        number_words (tuple): диапазон количества слов в строке (min, max)
        number_symbols (tuple): диапазон длины слов (min, max)
        
    Возвращает:
        generator: генератор строк
    """
    target_bytes = file_size * 1024**2  # Конвертируем МБ в байты
    generated_bytes = 0  # счётчик сгенерированных байт
    
    while generated_bytes < target_bytes:
        # Генерируем случайное количество слов
        word_count = randint(*number_words)
        # Создаем строку со случайными словами
        line = ' '.join(
            generate_random_word(randint(*number_symbols), type_symbols)
            for _ in range(word_count)
        ) + '\n'
        line_bytes = len(line.encode('utf-8'))  # получаем размер строки с учётом кодировки
        generated_bytes += line_bytes  # увеличиваем счётчик сгенерированных байт
        yield line  # возвращаем сгенерированную строку
        # Вывод прогресса
        progress = min(100, generated_bytes / target_bytes * 100)
        print(f'\rПрогресс: {progress:.1f}%', end='', flush=True)
    print(f"\nСгенерировано: {generated_bytes/1024**2:.2f} MB")
# Строки документации
print(lines_gen_fun_v2.__doc__)


    Генераторная функция для создания строк заданного размера
    
    Параметры:
        file_size (float): целевой размер файла в мегабайтах
        type_symbols (str): тип символов ('latin', 'digits' или 'cyrillic')
        number_words (tuple): диапазон количества слов в строке (min, max)
        number_symbols (tuple): диапазон длины слов (min, max)
        
    Возвращает:
        generator: генератор строк
    


Протестируем функцию `lines_gen_fun_v2` для различных значений
аргументов в предположении, что корректность вводимых данных
гарантируется.

In [70]:
print("\nТест 1: 0.1 MB латинских символов")
for line in lines_gen_fun_v2(0.1, 'latin', (3, 10), (2, 8)):
    pass


Тест 1: 0.1 MB латинских символов
Прогресс: 100.0%
Сгенерировано: 0.10 MB


In [75]:
print("\nТест 2: 0.5 MB цифр")
for line in lines_gen_fun_v2(0.5, 'digits', (5, 15), (1, 4)):
    pass


Тест 2: 0.5 MB цифр
Прогресс: 100.0%
Сгенерировано: 0.50 MB


In [77]:
print("\nТест 3: 1 MB кириллицы")
for line in lines_gen_fun_v2(1, 'cyrillic', (2, 8), (3, 10)):
    pass


Тест 3: 1 MB кириллицы
Прогресс: 100.0%
Сгенерировано: 1.00 MB


## Задание 9.3. Запись сгенерированных строк в текстовый файл

In [73]:
def file_gen(file_name: str, 
            file_size: int,
            type_symbols: str = 'latin',
            number_words: tuple = (10, 20),
            number_symbols: tuple = (5, 10)) -> None:
    start_time = process_time()  # Фиксируем начальное время выполнения
    
    total_bytes = 0  # Счетчик записанных байт
    try:
        with open(file_name, 'w', encoding='utf-8') as f:  # Открываем файл в режиме записи с UTF-8 кодировкой
            generator = lines_gen_fun_v2(file_size, type_symbols, number_words, number_symbols)  # Получаем генератор строк из lines_gen_fun_v2
            for line in generator:  # Поочередно записываем строки в файл
                f.write(line)  # Записываем текущую строку
                total_bytes += len(line.encode('utf-8'))  # Подсчитываем размер строки в байтах (с учетом кодировки)
        elapsed_time = process_time() - start_time  # Вычисляем затраченное процессорное время
        actual_size = total_bytes / (1024 ** 2)  # Переводим размер из байтов в мегабайты
        
        print(f"\nФайл '{file_name}' успешно создан")  # Выводим информацию о созданном файле
        print(f"Целевой размер: {file_size:.2f} MB")  # Целевой размер файла
        print(f"Фактический размер: {actual_size:.2f} MB")  # Реальный размер файла
        print(f"Время выполнения: {elapsed_time:.4f} сек")  # Время выполнения с точностью до 4 знаков
        
    except Exception as e:
        print(f"\nОшибка при создании файла: {e}")  # Обработка возможных ошибок при работе с файлом

In [82]:
print("Тест 1: 0.1 MB латинских символов")
file_gen('test_latin_small.txt', 0.1)

Тест 1: 0.1 MB латинских символов
Прогресс: 100.0%
Сгенерировано: 0.10 MB

Файл 'test_latin_small.txt' успешно создан
Целевой размер: 0.10 MB
Фактический размер: 0.10 MB
Время выполнения: 0.9844 сек


In [83]:
print("Тест 2: 1 MB цифр")
file_gen('test_digits_medium.txt', 1, 'digits', (5, 10), (2, 4))

Тест 2: 1 MB цифр
Прогресс: 100.0%
Сгенерировано: 1.00 MB

Файл 'test_digits_medium.txt' успешно создан
Целевой размер: 1.00 MB
Фактический размер: 1.00 MB
Время выполнения: 42.0000 сек


In [84]:
print("Тест 3: 5 MB кириллицы")
file_gen('test_cyrillic_large.txt', 5, 'cyrillic', (15, 25), (3, 8))

Тест 3: 5 MB кириллицы
Прогресс: 100.0%
Сгенерировано: 5.00 MB

Файл 'test_cyrillic_large.txt' успешно создан
Целевой размер: 5.00 MB
Фактический размер: 5.00 MB
Время выполнения: 32.4531 сек


In [85]:
print("Тест 4: 10 MB (смешанные параметры)")
file_gen('test_mixed_xlarge.txt', 10, 'latin', (20, 30), (10, 15))

Тест 4: 10 MB (смешанные параметры)
Прогресс: 100.0%
Сгенерировано: 10.00 MB

Файл 'test_mixed_xlarge.txt' успешно создан
Целевой размер: 10.00 MB
Фактический размер: 10.00 MB
Время выполнения: 46.3594 сек
